# ComfyUI + Z-Image Turbo - 2560x1440 生成

## 手順
1. ランタイム → GPU T4以上を選択
2. すべてのセルを実行
3. HuggingFace tokenを求められたら入力

In [ ]:
# GPU確認
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# HuggingFace token入力
print("HuggingFace tokenを入力してください")
print("(https://huggingface.co/settings/tokens で取得可能)")
HF_TOKEN = input("token: ")

In [ ]:
# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt -q

In [ ]:
# Z-Image Turboモデルをダウンロード
import os
import subprocess

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/models/configs", exist_ok=True)

from huggingface_hub import login
login(token=HF_TOKEN)

print("Downloading Z-Image Turbo model...")
from huggingface_hub import hf_hub_download

# メインモデル
hf_hub_download(
    repo_id="a-r-r-o-w/Z-Image-Turbo",
    filename="z-image-turbo-fp8-aio.safetensors",
    local_dir="/content/ComfyUI/models/checkpoints"
)

# 設定ファイル
hf_hub_download(
    repo_id="a-r-r-o-w/Z-Image-Turbo",
    filename="v1-inference.yaml",
    local_dir="/content/ComfyUI/models/configs"
)

print("Download complete!")

In [ ]:
# ComfyUI起動
import threading
import subprocess
import time

def run_comfyui():
    subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"])

t = threading.Thread(target=run_comfyui, daemon=True)
t.start()
print("ComfyUI starting...")
time.sleep(15)
print("Ready!")

In [ ]:
# 接続確認
import requests
try:
    r = requests.get("http://127.0.0.1:8188/system_stats", timeout=5)
    print("ComfyUI connected!")
except Exception as e:
    print(f"Waiting... {e}")
    time.sleep(20)
    r = requests.get("http://127.0.0.1:8188/system_stats")
    print("Connected!")

In [ ]:
# 画像生成（2560x1440）
import requests
import time
import json

workflow = {
    "1": {
        "inputs": {
            "ckpt_name": "z-image-turbo-fp8-aio.safetensors",
            "config_name": "v1-inference.yaml"
        },
        "class_type": "CheckpointLoader"
    },
    "2": {"inputs": {"text": "Professional stock photo of Japanese businessman in suit walking through Tokyo at night, neon lights, cinematic lighting, 4K quality, high detail", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "3": {"inputs": {"text": "blurry, low quality, distorted, watermark", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "4": {"inputs": {"width": 2560, "height": 1440, "batch_size": 1}, "class_type": "EmptyLatentImage"},
    "5": {"inputs": {"model": ["1", 0], "seed": 42, "steps": 8, "cfg": 1.0, "sampler_name": "euler", "scheduler": "normal", "positive": ["2", 0], "negative": ["3", 0], "latent_image": ["4", 0]}, "class_type": "KSampler"},
    "6": {"inputs": {"samples": ["5", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
    "7": {"inputs": {"images": ["6", 0], "filename_prefix": "result"}, "class_type": "SaveImage"}
}

print("Generating 2560x1440 image...")
result = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": workflow}).json()

if "prompt_id" in result:
    prompt_id = result['prompt_id']
    print(f"Prompt ID: {prompt_id}")
    
    for i in range(600):
        time.sleep(1)
        hist = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}").json()
        if prompt_id in hist and hist[prompt_id]['status']['completed']:
            print("Complete!")
            break
        if i % 30 == 0:
            print(f"Progress: {i}s...")
else:
    print(f"Error: {result}")

In [ ]:
# 画像ダウンロード
from google.colab import files
import glob
import os

output_files = glob.glob("/content/ComfyUI/output/result_*.png")
if output_files:
    latest = max(output_files, key=os.path.getmtime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No output found")